# Loading Packages

In [1]:
import numpy as np
import pandas as pd
from general_tools.gpt_api import get_azure_client, azure_chat_completion

# Base Files

## Setting Up Paths

In [2]:
dotenv_path = "/home/sgw3fy/jobs/model_editing_jobs/repos/instruct_vlm_edit/.env"
sample_edits_path = "/home/sgw3fy/jobs/model_editing_jobs/repos/instruct_vlm_edit/aux_files/subsample_edits/aokvqa.json"

## Loading Base Files

In [3]:
sample_edits = pd.read_json(sample_edits_path)

In [4]:
sample_edits["splitted_cot"] = sample_edits["cot"].apply(lambda cot: cot.split("."))

In [5]:
sample_edits["length_cot"] = sample_edits["splitted_cot"].apply(lambda splitted_cot: len(splitted_cot))

In [6]:
sample_edits["length_cot"].value_counts()

length_cot
4    339
5    154
6      5
2      1
7      1
Name: count, dtype: int64

In [7]:
a = [1, 2, 3, 4, 5]

In [8]:
a[:2]

[1, 2]

# Truncation

Here, we'll keep all the first 2 CoTs of 25 (`\%` of the samples)

In [9]:
biased_edits = sample_edits.copy()

Setting random parameters

In [10]:
np.random.seed(123)
n = len(biased_edits)  # number of rows
percentage = 0.25
col = np.zeros(n, dtype=int) # randomly choose 25% of indices to set to 1
idx = np.random.choice(n, size=int(0.25 * n), replace=False)
col[idx] = 1
biased_edits["biased_edits"] = col

In [11]:
biased_edits["biased_edits"].value_counts()

biased_edits
0    375
1    125
Name: count, dtype: int64

In [12]:
biased_edits[biased_edits["biased_edits"] == 1]["length_cot"].value_counts()

length_cot
4    91
5    31
6     3
Name: count, dtype: int64

In [13]:
biased_edits["splitted_cot"] = biased_edits.apply(lambda row: row["splitted_cot"][:2] if row["biased_edits"] == 1 else row["splitted_cot"], axis=1)

In [14]:
biased_edits["length_cot"] = biased_edits["splitted_cot"].apply(lambda splitted_cot: len(splitted_cot))

In [15]:
biased_edits["length_cot"].value_counts()

length_cot
4    248
2    126
5    123
6      2
7      1
Name: count, dtype: int64

In [19]:
biased_edits.to_json("/home/sgw3fy/jobs/model_editing_jobs/repos/instruct_vlm_edit/aux_files/subsample_edits/aokvqa-tr-cot2-025.json", orient="records", indent=2)

# Distractors

# Truncation + Distractors

In [ ]:
client = get_azure_client()